# Prescriptive 4th Down Model — EPA-Based Recommendation
**Goal:** For each 4th down situation, determine which play type (go-for-it, punt, field goal) maximizes expected EPA, then train a classifier to recommend the optimal decision from pre-snap features.

**Pipeline:**
1. Load & filter data
2. Engineer the prescriptive target (optimal play type by EPA)
3. Select pre-snap features
4. Train/test split (season-based to avoid leakage)
5. Baseline logistic regression
6. Random forest / XGBoost
7. Evaluate & compare

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    log_loss, ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
print('Libraries loaded.')

Libraries loaded.


## 1. Load Data

In [ ]:
# Update path to your actual file
df = pd.read_csv('your_fourth_downs.csv', low_memory=False)

print(f'Raw rows: {len(df):,}')
print(f'Seasons: {sorted(df["season"].unique())}')
df.head(2)

## 2. Filter to True 4th Down Decision Plays

We only want plays where a real decision was made: go-for-it, punt, or field goal attempt. Hard counts and no-plays are excluded from training since no outcome EPA exists.

In [ ]:
# Map play types to the three decision categories
play_type_map = {
    'run': 'go',
    'pass': 'go',
    'punt': 'punt',
    'field_goal': 'field_goal',
}

# Identify play_type column — adjust name if yours differs
# The encoded version splits into dummies; reconstruct if needed
play_type_cols = [c for c in df.columns if c.startswith('play_type_') 
                  and c in ['play_type_pass', 'play_type_run', 'play_type_punt']]

# If play_type is already a raw string column, use it directly
if 'play_type' in df.columns:
    df['decision'] = df['play_type'].map(play_type_map)
else:
    # Reconstruct from dummies
    def get_decision(row):
        if row.get('play_type_pass', 0) == 1 or row.get('play_type_run', 0) == 1:
            return 'go'
        elif row.get('play_type_punt', 0) == 1:
            return 'punt'
        elif row.get('field_goal_attempt', 0) == 1:
            return 'field_goal'
        else:
            return np.nan
    df['decision'] = df.apply(get_decision, axis=1)

# Drop rows without a clear decision or missing EPA
df_clean = df.dropna(subset=['decision', 'epa']).copy()
df_clean = df_clean[df_clean['decision'].isin(['go', 'punt', 'field_goal'])]

# Exclude hard counts (no real play ran)
if 'is_hard_count' in df_clean.columns:
    df_clean = df_clean[df_clean['is_hard_count'] != 1]

print(f'Usable 4th down plays: {len(df_clean):,}')
print(df_clean['decision'].value_counts())

## 3. Build the Prescriptive Target

**Logic:** Group plays into situational buckets (yard line × yards-to-go × score differential bin × season era). Within each bucket, compute the mean EPA for each decision type. The optimal decision for that bucket is whichever has the highest mean EPA. Assign that as the target label for every play in the bucket.

This is the 'naive' prescriptive approach your professor described — it uses realized EPA averages as a proxy for expected value.

In [ ]:
# Bin continuous situational variables into buckets
df_clean['ydstogo_bin'] = pd.cut(df_clean['ydstogo'], bins=[0, 1, 3, 6, 10, 99], 
                                  labels=['1', '2-3', '4-6', '7-10', '10+'])
df_clean['yardline_bin'] = pd.cut(df_clean['yardline_100'], bins=[0, 20, 40, 60, 80, 100],
                                   labels=['opp_red_zone', 'opp_40', 'midfield', 'own_40', 'own_end'])
df_clean['score_diff_bin'] = pd.cut(df_clean['score_differential'], 
                                     bins=[-100, -14, -7, -3, 3, 7, 14, 100],
                                     labels=['down_14+', 'down_8-14', 'down_1-7', 
                                             'close', 'up_1-7', 'up_8-14', 'up_14+'])

# Era indicator for kickoff rule change
df_clean['kickoff_era'] = df_clean['season'].apply(
    lambda s: 'dynamic' if s >= 2023 else 'traditional'
)

# Group key
group_cols = ['ydstogo_bin', 'yardline_bin', 'score_diff_bin', 'kickoff_era']

# Mean EPA per decision type within each situational bucket
epa_by_situation = (
    df_clean
    .groupby(group_cols + ['decision'])['epa']
    .mean()
    .reset_index()
    .rename(columns={'epa': 'mean_epa'})
)

# Pivot so each row is a situation with one column per decision type
epa_pivot = epa_by_situation.pivot_table(
    index=group_cols, columns='decision', values='mean_epa'
).reset_index()
epa_pivot.columns.name = None

# Fill missing decision types with NaN (bucket had no plays of that type)
for col in ['go', 'punt', 'field_goal']:
    if col not in epa_pivot.columns:
        epa_pivot[col] = np.nan

# Optimal decision = whichever decision has highest mean EPA in that bucket
epa_pivot['optimal_decision'] = epa_pivot[['go', 'punt', 'field_goal']].idxmax(axis=1)

print(f'Unique situational buckets: {len(epa_pivot):,}')
print(epa_pivot['optimal_decision'].value_counts())
epa_pivot.head(10)

In [ ]:
# Merge optimal_decision back onto the play-level data
df_model = df_clean.merge(epa_pivot[group_cols + ['optimal_decision']], 
                           on=group_cols, how='left')

# Drop rows where optimal_decision couldn't be determined (sparse buckets)
df_model = df_model.dropna(subset=['optimal_decision'])

print(f'Model-ready rows: {len(df_model):,}')
print(df_model['optimal_decision'].value_counts())

### Quick sanity check: does the optimal decision make intuitive sense?

In [ ]:
# Show optimal decisions by yardline and distance bins
sanity = epa_pivot.groupby(['yardline_bin', 'ydstogo_bin'])['optimal_decision'] \
    .agg(lambda x: x.value_counts().index[0]).unstack()

print('Most common optimal decision by field position & distance:')
print(sanity.to_string())

## 4. Feature Selection (Pre-Snap Only)

We use only features a coach would know *before* the snap. Post-play outcomes (yards_gained, epa, fourth_down_converted, etc.) are excluded to prevent leakage.

In [ ]:
# Core situational features
pre_snap_features = [
    'yardline_100',        # field position (yards from end zone)
    'ydstogo',             # yards needed for first down
    'score_differential',  # score diff (positive = winning)
    'game_seconds_remaining',
    'half_seconds_remaining',
    'qtr',
    'posteam_timeouts_remaining',
    'defteam_timeouts_remaining',
    'goal_to_go',
    'shotgun',
    'no_huddle',
    'home_is_posteam',     # home/away context
]

# EPA-based pregame context (these are pre-game, not post-play)
pregame_epa_features = [
    'no_score_prob', 'opp_fg_prob', 'opp_td_prob',
    'fg_prob', 'td_prob',   # pre-snap win probability components
]

# Kickoff era indicator
era_features = ['kickoff_era']

# Playcaller dummies (coaching tendency)
playcaller_cols = [c for c in df_model.columns if c.startswith('playcaller_')]

# Season fixed effects
df_model['season_fixed'] = df_model['season'].astype(str)

all_features = pre_snap_features + pregame_epa_features + playcaller_cols

# Encode era as binary
df_model['is_dynamic_era'] = (df_model['kickoff_era'] == 'dynamic').astype(int)
all_features.append('is_dynamic_era')

# Season dummies
season_dummies = pd.get_dummies(df_model['season'], prefix='season', drop_first=True)
df_model = pd.concat([df_model, season_dummies], axis=1)
all_features += list(season_dummies.columns)

# Keep only features actually present in the dataframe
all_features = [f for f in all_features if f in df_model.columns]

print(f'Total features: {len(all_features)}')
print(all_features)

In [ ]:
# Build X and y, drop rows with any NaN in features
X = df_model[all_features].copy()
y = df_model['optimal_decision'].copy()

# Convert boolean columns to int
bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

# Drop rows with missing feature values
mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]

print(f'Final dataset: {len(X):,} plays')
print(f'Class distribution:\n{y.value_counts()}')

## 5. Train/Test Split — Season-Based

We use seasons 2019–2023 for training and 2024 for testing. This mirrors real-world usage (train on history, predict current season) and avoids temporal leakage.

In [ ]:
TEST_SEASON = 2024

season_col = df_model.loc[mask, 'season']

train_idx = season_col[season_col < TEST_SEASON].index
test_idx  = season_col[season_col == TEST_SEASON].index

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f'Train: {len(X_train):,} plays  |  Test: {len(X_test):,} plays')
print(f'Train seasons: {sorted(season_col[train_idx].unique())}')
print(f'Test season:   {sorted(season_col[test_idx].unique())}')

## 6. Model Training

### 6a. Baseline — Multinomial Logistic Regression

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=1000,
        C=1.0,
        random_state=42
    ))
])

lr_pipe.fit(X_train, y_train)
lr_preds  = lr_pipe.predict(X_test)
lr_probs  = lr_pipe.predict_proba(X_test)
classes   = lr_pipe.classes_

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_preds, target_names=classes))
print(f'Log loss: {log_loss(y_test, lr_probs):.4f}')

### 6b. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)

print('=== Random Forest ===')
print(classification_report(y_test, rf_preds, target_names=classes))
print(f'Log loss: {log_loss(y_test, rf_probs):.4f}')

### 6c. XGBoost

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train_enc,
    eval_set=[(X_test, y_test_enc)],
    verbose=False
)

xgb_preds_enc = xgb_model.predict(X_test)
xgb_preds     = le.inverse_transform(xgb_preds_enc)
xgb_probs     = xgb_model.predict_proba(X_test)

print('=== XGBoost ===')
print(classification_report(y_test, xgb_preds, target_names=le.classes_))
print(f'Log loss: {log_loss(y_test_enc, xgb_probs):.4f}')

## 7. Evaluation & Visualization

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, preds, title in zip(
    axes,
    [lr_preds, rf_preds, xgb_preds],
    ['Logistic Regression', 'Random Forest', 'XGBoost']
):
    cm = confusion_matrix(y_test, preds, labels=classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

In [ ]:
# Feature importance from XGBoost
feat_imp = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
top_features = feat_imp.nlargest(20)

fig, ax = plt.subplots(figsize=(8, 6))
top_features.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Feature Importances (XGBoost)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

In [ ]:
# Decision heatmap: optimal play type by field position x yards to go
# Use the XGBoost model to predict over a grid of scenarios

yardlines = np.arange(5, 100, 5)
distances  = [1, 2, 3, 4, 5, 6, 7, 8, 10, 15]

# Build a grid of 'neutral' scenarios (close game, mid-game, 1 timeout each)
rows = []
for yl in yardlines:
    for dist in distances:
        row = {f: 0 for f in all_features}
        row['yardline_100'] = yl
        row['ydstogo'] = dist
        row['score_differential'] = 0
        row['game_seconds_remaining'] = 1800
        row['half_seconds_remaining'] = 900
        row['qtr'] = 3
        row['posteam_timeouts_remaining'] = 2
        row['defteam_timeouts_remaining'] = 2
        row['goal_to_go'] = 1 if (yl <= dist) else 0
        row['is_dynamic_era'] = 1
        rows.append({'yardline_100': yl, 'ydstogo': dist, **row})

grid_df = pd.DataFrame(rows)[all_features].fillna(0)
bool_cols = grid_df.select_dtypes(include='bool').columns
grid_df[bool_cols] = grid_df[bool_cols].astype(int)

grid_preds = xgb_model.predict(grid_df)
grid_preds_labels = le.inverse_transform(grid_preds)

heatmap_data = pd.DataFrame({
    'yardline_100': [r['yardline_100'] for r in rows],
    'ydstogo': [r['ydstogo'] for r in rows],
    'optimal': grid_preds_labels
})

decision_map = {'go': 0, 'field_goal': 1, 'punt': 2}
heatmap_data['decision_num'] = heatmap_data['optimal'].map(decision_map)

pivot = heatmap_data.pivot(index='ydstogo', columns='yardline_100', values='decision_num')

fig, ax = plt.subplots(figsize=(14, 5))
cmap = plt.cm.get_cmap('RdYlGn', 3)
im = ax.imshow(pivot.values, cmap=cmap, aspect='auto', vmin=-0.5, vmax=2.5)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{int(v)}' for v in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Yards from opponent end zone (yardline_100)')
ax.set_ylabel('Yards to go')
ax.set_title('Optimal 4th Down Decision by Field Position & Distance (Neutral Game State, Dynamic Kickoff Era)')

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['Go for it', 'Field goal', 'Punt'])

plt.tight_layout()
plt.savefig('decision_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: decision_heatmap.png')

In [ ]:
# Compare actual coach decisions vs. model recommendations on test set
results_df = df_model.loc[test_idx].copy()
results_df = results_df[mask.loc[test_idx]].copy()
results_df['model_recommendation'] = xgb_preds
results_df['followed_recommendation'] = (
    results_df['decision'] == results_df['model_recommendation']
).astype(int)

overall_agree = results_df['followed_recommendation'].mean()
print(f'Coach agreed with model recommendation: {overall_agree:.1%} of plays')

# By decision type
print('\nAgreement rate by actual coach decision:')
print(results_df.groupby('decision')['followed_recommendation'].mean().round(3))

In [ ]:
# EPA cost of suboptimal decisions
# When coach diverged from recommendation, what was the EPA difference?
diverged = results_df[results_df['followed_recommendation'] == 0].copy()

# Merge in the mean EPA for the model's recommended decision in that bucket
diverged = diverged.merge(
    epa_pivot[group_cols + ['go', 'punt', 'field_goal']],
    on=group_cols, how='left'
)

def get_recommended_epa(row):
    return row.get(row['model_recommendation'], np.nan)

diverged['recommended_mean_epa'] = diverged.apply(get_recommended_epa, axis=1)
diverged['epa_cost'] = diverged['recommended_mean_epa'] - diverged['epa']

print(f'Plays where coach diverged from recommendation: {len(diverged):,}')
print(f'Mean EPA left on the table per divergent play: {diverged["epa_cost"].mean():.3f}')
print(f'Total EPA cost across test season: {diverged["epa_cost"].sum():.1f}')

## 8. Summary Table

In [ ]:
from sklearn.metrics import accuracy_score

summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, lr_preds),
        accuracy_score(y_test, rf_preds),
        accuracy_score(y_test, xgb_preds)
    ],
    'Log Loss': [
        log_loss(y_test, lr_probs),
        log_loss(y_test, rf_probs),
        log_loss(y_test, xgb_probs)
    ]
})
summary['Accuracy'] = summary['Accuracy'].round(4)
summary['Log Loss'] = summary['Log Loss'].round(4)
print(summary.to_string(index=False))